# Lab 28 — Full Platform Integration Sprint
**GPU: T4 | Internet: ON**

## Cell 1 — Install

In [ ]:
!pip install -q vllm fastapi uvicorn mlflow sentence-transformers requests pyngrok httpx

import torch, os, glob, subprocess
print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB')
print(f'Compute: {torch.cuda.get_device_capability(0)}')
from pyngrok import ngrok
print('pyngrok OK')


## Cell 2 — Ngrok Auth

In [ ]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok

try:
    NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    print(f'Token loaded ({len(NGROK_TOKEN)} chars)')
except:
    NGROK_TOKEN = ""
    if not NGROK_TOKEN:
        raise SystemExit("Cần NGROK_AUTHTOKEN trong Kaggle Secrets")

ngrok.set_auth_token(NGROK_TOKEN)
print('Ngrok authenticated')


## Cell 3 — vLLM
> Fix: LIBRARY_PATH stub cho flashinfer linker (`-lcuda`)

In [ ]:
import subprocess, threading, time, requests, os, glob

# ── Fix libcuda stub cho flashinfer linker ──────────────────────
os.makedirs('/tmp/cuda_stubs', exist_ok=True)
stub = '/tmp/cuda_stubs/libcuda.so'
if not os.path.exists(stub):
    for src in [
        '/usr/lib/x86_64-linux-gnu/libcuda.so.1',
        '/usr/lib/x86_64-linux-gnu/libcuda.so',
        *glob.glob('/usr/local/cuda/lib64/libcudart.so*'),
    ]:
        if os.path.exists(src):
            os.symlink(src, stub)
            print(f'libcuda stub: {stub} → {src}')
            break
    else:
        print('WARNING: libcuda source not found')

# LIBRARY_PATH cho linker (ld), LD_LIBRARY_PATH cho runtime
os.environ['LIBRARY_PATH'] = '/tmp/cuda_stubs:' + os.environ.get('LIBRARY_PATH','')
os.environ['LD_LIBRARY_PATH'] = '/tmp/cuda_stubs:' + os.environ.get('LD_LIBRARY_PATH','')

# Kill old vLLM
subprocess.run(['pkill','-f','vllm.entrypoints'], capture_output=True)
time.sleep(2)

def run_vllm():
    env = os.environ.copy()
    subprocess.run([
        'python','-m','vllm.entrypoints.openai.api_server',
        '--model','Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4',
        '--port','8001','--max-model-len','2048',
        '--gpu-memory-utilization','0.90',
        '--host','0.0.0.0','--enforce-eager',
    ], env=env)

print('Starting vLLM (~4 min)...')
threading.Thread(target=run_vllm, daemon=True).start()

for i in range(30):
    time.sleep(10)
    try:
        r = requests.get('http://localhost:8001/v1/models', timeout=3)
        if r.status_code == 200:
            print(f'vLLM ready after {(i+1)*10}s: {[m["id"] for m in r.json().get("data",[])]}')
            break
    except: pass
    print(f'  Loading... {(i+1)*10}s')
else:
    print('vLLM timeout')


## Cell 4 — Unified Proxy
> Embedding + proxy vLLM → **1 port duy nhất (8000)**

In [ ]:
from fastapi import FastAPI, Request
from sentence_transformers import SentenceTransformer
import uvicorn, threading, httpx, asyncio

# ── Embedding model ──────────────────────────────────────────────
print('Loading BAAI/bge-small-en-v1.5...')
embed_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
print('Embedding model loaded')

# ── Unified proxy on port 8000 ───────────────────────────────────
# /embed  → local embedding
# /v1/*   → vLLM on port 8001
proxy = FastAPI(title='Lab28 Unified Gateway')

@proxy.post('/embed')
async def embed(data: dict):
    texts = data.get('texts', [])
    if not texts:
        return {'embeddings':[], 'error':'No texts'}
    return {'embeddings': embed_model.encode(texts, normalize_embeddings=True).tolist(), 'count': len(texts)}

@proxy.get('/health')
def health():
    return {'status':'ok', 'services':['embed','vllm-proxy']}

@proxy.api_route('/v1/{path:path}', methods=['GET','POST','PUT','DELETE'])
async def proxy_vllm(path: str, request: Request):
    body = await request.body()
    async with httpx.AsyncClient(timeout=120) as client:
        r = await client.request(
            method=request.method,
            url=f'http://localhost:8001/v1/{path}',
            headers=dict(request.headers),
            content=body,
        )
    from fastapi.responses import Response
    return Response(content=r.content, status_code=r.status_code,
                    media_type=r.headers.get('content-type','application/json'))

threading.Thread(
    target=lambda: uvicorn.run(proxy, host='0.0.0.0', port=8000, log_level='warning'),
    daemon=True
).start()
print('Unified proxy started on port 8000')
print('  /embed  → embedding (local)')
print('  /v1/*   → vLLM proxy (port 8001)')


## Cell 5 — 1 Ngrok Tunnel
> Ngrok free = 1 tunnel. Cả VLLM_URL và EMBED_URL dùng cùng BASE_URL

In [ ]:
from pyngrok import ngrok
import time, requests

ngrok.kill()
time.sleep(1)

# CHỈ CẦN 1 TUNNEL duy nhất
tunnel = ngrok.connect(8000, 'http')
BASE_URL = tunnel.public_url.replace('http://','https://',1)

print(f'BASE URL: {BASE_URL}')
print()
print('Paste vào .env local:')
print(f'  VLLM_NGROK_URL={BASE_URL}')
print(f'  EMBED_NGROK_URL={BASE_URL}')
print()
print('API endpoints:')
print(f'  vLLM chat: {BASE_URL}/v1/chat/completions')
print(f'  Embed:     {BASE_URL}/embed')
print(f'  Health:    {BASE_URL}/health')


## Cell 6 — MLflow

In [ ]:
import mlflow
mlflow.set_tracking_uri('./mlruns')
mlflow.set_experiment('lab28-integration')
with mlflow.start_run(run_name='lab28-v1') as run:
    mlflow.log_param('model','Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4')
    mlflow.log_param('embed_model','BAAI/bge-small-en-v1.5')
    mlflow.log_param('enforce_eager', True)
    mlflow.set_tag('base_url', BASE_URL)
    mlflow.set_tag('lab','lab28')
print(f'MLflow run_id={run.info.run_id}')


## Cell 7 — Test

In [ ]:
import requests

print('='*55)
r = requests.get(f'{BASE_URL}/health', timeout=10)
print(f'[health] {r.status_code}: {r.json()}')

print('\n[embed]')
r = requests.post(f'{BASE_URL}/embed', json={'texts':['hello world','AI test']}, timeout=30)
if r.status_code == 200:
    d = r.json()
    print(f'  count={d["count"]}, dim={len(d["embeddings"][0])}')
else:
    print(f'  ERROR {r.status_code}')

print('\n[vLLM]')
try:
    r = requests.post(f'{BASE_URL}/v1/chat/completions', json={
        'model':'Qwen/Qwen2.5-7B-Instruct-GPTQ-Int4',
        'messages':[{'role':'user','content':'Reply: Lab28 OK'}],
        'max_tokens':10
    }, timeout=60)
    if r.status_code == 200:
        print(f'  {r.json()["choices"][0]["message"]["content"]}')
    else:
        print(f'  HTTP {r.status_code}: {r.text[:200]}')
except Exception as e:
    print(f'  Error: {e}')

print('='*55)
print(f'VLLM_NGROK_URL={BASE_URL}')
print(f'EMBED_NGROK_URL={BASE_URL}')


## Cell 8 — Keep Alive

In [ ]:
import time, requests
print(f'Active URL: {BASE_URL}')
print('Keep alive... (Interrupt to stop)')
i = 0
while True:
    time.sleep(300)
    i += 1
    try: requests.get('http://localhost:8001/v1/models', timeout=2)
    except: pass
    print(f'  [{i*5}min] {BASE_URL}')
